# Effect of the Thomson scattering noise model

The seven ground truths inferred with MAP + Laplace under two Thomson scattering noise
models, compared on the inferred profiles and their uncertainty:
- `poisson`: photon counting only, `sigma_i = sqrt(D_i)`.
- `electronic`: the W7-X detector noise curve, `sigma_i / I_i = a exp(-b I_i) + c`
  (Raak 2024).

Each model sets both the noise drawn on the synthetic Thomson signals and the width the
likelihood uses. The two runs of a truth share the noise seed and the interferometer
readings.

```
python run_study.py --workers 7
```

writes `tables/`; this notebook reads them and writes `figures/`.

In [ ]:
import json
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from plasma_state_estimation_for_diagnostic_placement_optimization.bayesian_inference import (  # noqa: E501
    desc_truth_types,
    inference_model_utils,
    profile_parametrization_types,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.inverse_mapping import (  # noqa: E501
    inverse_map_utils,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.mhd_equilibrium_forward_models import (  # noqa: E501
    dataset_utils,
    model_storage_utils,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.utils import (  # noqa: E501
    inference_metrics_utils,
    inference_store_utils,
    profile_table_utils,
    pubfig,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.utils import (  # noqa: E501
    plots_plasma_state_estimation as plots,
)

pubfig.apply_style()
pd.set_option("display.max_colwidth", None)

PACKAGE_DIRECTORY = pathlib.Path(plots.__file__).parents[2]
NOTEBOOK_DIRECTORY = (
    PACKAGE_DIRECTORY / "results_plots_tables" / "results" / "plasma_state_estimation"
) / "prior_likelihood_posterior"
FIGURES_DIRECTORY = NOTEBOOK_DIRECTORY / "figures"
TABLES_DIRECTORY = NOTEBOOK_DIRECTORY / "tables"
DATA_DIRECTORY = PACKAGE_DIRECTORY.parents[1] / "data"
MODELS_DIRECTORY = (
    PACKAGE_DIRECTORY / "mhd_equilibrium_forward_models" / "trained_models"
)
PARQUET = DATA_DIRECTORY / f"{dataset_utils.DEFAULT_DATASET}.parquet"
SETTINGS = DATA_DIRECTORY / f"{dataset_utils.DEFAULT_DATASET}_settings.json"

configuration = json.loads(
    (TABLES_DIRECTORY / "prior_likelihood_posterior_configuration.json").read_text()
)
CASES = tuple(configuration["inference_cases"])
LABELS = {"poisson": "Poisson", "electronic": "electronic"}
profiles = pd.read_csv(TABLES_DIRECTORY / "prior_likelihood_posterior_profiles.csv")
summary = pd.read_csv(TABLES_DIRECTORY / "prior_likelihood_posterior_summary.csv")
thomson_points = pd.read_csv(
    TABLES_DIRECTORY / "prior_likelihood_posterior_thomson_points.csv"
)

# The runs the tables were written from, one per truth and noise model.
runs = configuration["runs"]
loaded = {
    (run["profile_id"], run["inference_case"]): inference_store_utils.load_inference(
        inference_store_utils.INFERENCE_STORE, run["inference_id"]
    )
    for run in runs
}
truths = list(dict.fromkeys(run["profile_id"] for run in runs))
print(f"{len(loaded)} runs over {len(truths)} truths")

In [ ]:
record = loaded[(truths[0], CASES[-1])][0]
placement = inference_store_utils.load_placement(
    inference_store_utils.PLACEMENT_STORE, record.diagnostics_placement_id
)
seeds = sorted({run["noise_seed"] for run in runs})
pd.Series(
    {
        "ground truths": ", ".join(truths),
        "noise draws per truth": f"1 (seeds {seeds[0]} to {seeds[-1]})",
        "Thomson noise models": ", ".join(CASES),
        "Thomson scattering volumes": len(placement.thomson_geometry.volume_positions),
        "interferometer lines of sight": len(placement.interferometer_chords),
        "interferometer sigma_IF [m^-2]": f"{record.settings.interferometer_sigma:.3g}",
        "calibration correction per volume": (
            f"log-uniform on [1/N, N], N = {record.settings.thomson_calibration_bound}"
        ),
        "profile family": "two-power, center * (1 - rho^alpha)^beta",
        "MAP starts": f"{record.settings.map.n_starts}, drawn from the priors",
        "posterior": "Laplace Gaussian at the best converged MAP start",
        "draws per band": configuration["n_posterior_samples"],
    },
    name="setting",
).to_frame()

The prior band, from drawing every profile parameter from its box, with the seven
truths over it.

In [ ]:
figure = plots.plot_prior_bands(profiles, prior_case=configuration["prior_case"])
pubfig.save(figure, str(FIGURES_DIRECTORY / "prior_profile"))
display(figure)
plt.close(figure)

## 1. Did the MAP converge?

L-BFGS from one prior-drawn start, as in `mcmc_benchmark`; a start has converged when it
stops before the 1000-iteration cap. `chi2 / dof` near one means the noise model's width
matches the noise it drew.

The figures show each profile parameter over its true value along the iterations (mean
and 5 to 95% quantiles over the seven truths); converged MAPs flatten before the end.

In [ ]:
map_table = pd.DataFrame(runs)
map_table["noise model"] = map_table["inference_case"].map(LABELS)
map_table["chi2 / dof"] = map_table["chi_squared"] / map_table["n_degrees_of_freedom"]
map_table = map_table[
    [
        "profile_id",
        "noise model",
        "n_map_starts_converged",
        "n_map_modes",
        "map_gradient_inf_norm",
        "chi2 / dof",
    ]
].set_axis(
    [
        "truth",
        "noise model",
        "converged starts",
        "distinct minima",
        "|grad U|_inf at MAP",
        "chi2 / dof",
    ],
    axis=1,
)
display(map_table.round({"|grad U|_inf at MAP": 4, "chi2 / dof": 2}))

truth_profiles = {
    truth: desc_truth_types.read_desc_truth(truth, PARQUET, SETTINGS).profiles
    for truth in truths
}
# One start per truth, so the quantiles run over the truths of each noise model.
for case in CASES:
    figure = plots.plot_map_convergence(
        [(*loaded[(truth, case)], truth_profiles[truth]) for truth in truths],
        label=LABELS[case],
    )
    name = f"noise_model_convergence_{case}"
    pubfig.save(figure, str(FIGURES_DIRECTORY / name))
    display(figure)
    plt.close(figure)

## 2. Effect on the inferred profiles

One figure per truth and noise model: the truth, the initial guess, the MAP with its
Laplace 5 to 95% band, and the per-volume Thomson estimates with the uncertainty that
noise model gives them.

In [ ]:
for truth in truths:
    truth_state = desc_truth_types.read_desc_truth(truth, PARQUET, SETTINGS)
    for case in CASES:
        record = loaded[(truth, case)][0]
        figure = plots.plot_method_band_comparison(
            profile_table_utils.profile_band_table(record, "map_laplace"),
            ("map_laplace",),
            thomson_points=profile_table_utils.thomson_points_table(
                record, truth_state
            ),
            title=f"{truth}, {LABELS[case]}",
            filled_bands=True,
        )
        name = f"noise_model_profile_{truth}_{case}"
        pubfig.save(figure, str(FIGURES_DIRECTORY / name))
        display(figure)
        plt.close(figure)

## 3. Effect on the uncertainty

The integrated band width `U = integral sigma(rho) drho` as a percentage of the prior's:
100% means the data added nothing, smaller is a stronger constraint.

In [ ]:
positions = np.arange(len(truths))
figure, axes_array = pubfig.subplots(1, 2, width="double", aspect=0.85)
for axes, symbol, ratio_label in zip(
    axes_array, ("ne", "Te"), (r"$R_{n_e}$", r"$R_{T_e}$")
):
    # Dashed over solid, so both noise models stay visible where they agree.
    for case, marker, dashes, color in zip(
        CASES, ("o", "s"), ((0, ()), (0, (5.0, 1.6))), pubfig.PALETTE
    ):
        ratios = (
            summary.loc[summary["inference_case"] == case]
            .set_index("profile_id")[f"R_{symbol}"]
            .reindex(truths)
            .to_numpy()
        )
        axes.plot(
            positions,
            ratios,
            marker,
            ls=dashes,
            color=color,
            lw=1.1,
            label=LABELS[case],
        )
    axes.axhline(1.0, color="0.55", ls=":", lw=0.9, zorder=0)
    axes.set_yscale("log")
    # Headroom above the reference, which otherwise lands on the top spine.
    axes.set_ylim(top=1.4 * max(float(axes.get_ylim()[1]), 1.0))
    axes.set_xticks(positions)
    axes.set_xticklabels([truth.replace("_", " ") for truth in truths], rotation=45)
    axes.set_ylabel(ratio_label)
axes_array[0].legend(fontsize=6)
pubfig.save(figure, str(FIGURES_DIRECTORY / "uncertainty_reduction_summary"))
display(figure)
plt.close(figure)

reduction = summary.loc[summary["inference_case"].isin(CASES)].pivot(
    index="profile_id", columns="inference_case", values=["R_ne", "R_Te"]
)
reduction = 100.0 * reduction.rename(
    columns={"R_ne": "ne", "R_Te": "Te", **LABELS}
).rename_axis(index="truth", columns=["profile", "noise model"])
reduction = reduction[
    pd.MultiIndex.from_product([("ne", "Te"), [LABELS[case] for case in CASES]])
]
reduction = pd.concat([reduction, reduction.agg(["mean", "std"])])
reduction.round(2)

## 4. Summary

For `ne` and `Te`: the core error at `rho = 0` (percent of the true value), the RMSE over
the radius (percent of the true peak) and the median band half-width (percent of the
local value). The last two rows are the mean and standard deviation over the truths.

In [ ]:
rows = []
for (truth, case), (record, _) in loaded.items():
    accuracy = {
        item.name: item for item in inference_metrics_utils.profile_accuracy(record)
    }
    for symbol, name in (("ne", "electron_density"), ("Te", "electron_temperature")):
        rows.append(
            {
                "truth": truth,
                "profile": symbol,
                "noise model": LABELS[case],
                "core error [%]": accuracy[name].axis_error_percent,
                "RMSE [%]": accuracy[name].error_percent,
                "band half-width [%]": accuracy[name].uncertainty_percent,
            }
        )
metrics = pd.DataFrame(rows).pivot(index="truth", columns=["profile", "noise model"])
metrics = metrics.reorder_levels([1, 0, 2], axis=1)[
    pd.MultiIndex.from_product(
        [
            ("ne", "Te"),
            ("core error [%]", "RMSE [%]", "band half-width [%]"),
            [LABELS[case] for case in CASES],
        ]
    )
]
metrics = pd.concat([metrics, metrics.agg(["mean", "std"])])
metrics.round(2)

## 5. Why the Poisson fits fail

The surrogate's flux-surface error at each scattering volume against the Thomson
uncertainty there. The ground-truth profiles are evaluated at the `rho` that the DESC
equilibrium and the surrogate give each volume, so their difference is the surrogate's
error alone; `sigma` is the 1-sigma width of each volume's own Thomson estimate (the
points of section 2), as a percentage of the local value. The `ne` and `Te` errors
coincide, since both ground-truth profiles are proportional to `sqrt(p)`.

In [ ]:
PROFILE_FUNCTIONS = {
    "ne": profile_parametrization_types.electron_density,
    "Te": profile_parametrization_types.electron_temperature,
}

volume_tables = []
# The geometry is the same under both noise models, so one run per truth suffices.
for run in runs:
    if run["inference_case"] != "poisson":
        continue
    truth_state = desc_truth_types.read_desc_truth(run["profile_id"], PARQUET, SETTINGS)
    surrogate = model_storage_utils.load_surrogate(
        MODELS_DIRECTORY,
        run["surrogate_name"],
        DATA_DIRECTORY,
        machine_scale=truth_state.machine_scale,
    )
    volume_positions = inference_store_utils.load_placement(
        inference_store_utils.PLACEMENT_STORE, run["diagnostics_placement_id"]
    ).thomson_geometry.volume_positions
    rho_desc = np.asarray(
        inverse_map_utils.implicit_flux_solve(
            truth_state.coefficients, volume_positions
        )[0]
    )
    surrogate_coefficients = inference_model_utils.equilibrium_coefficients(
        surrogate, truth_state.profiles, None
    )
    rho_surrogate = np.asarray(
        inverse_map_utils.implicit_flux_solve(surrogate_coefficients, volume_positions)[
            0
        ]
    )
    for symbol, profile_function in PROFILE_FUNCTIONS.items():
        exact = np.asarray(profile_function(truth_state.profiles, rho_desc))
        shifted = np.asarray(profile_function(truth_state.profiles, rho_surrogate))
        volume_table = pd.DataFrame(
            {
                "truth": run["profile_id"],
                "profile": symbol,
                "volume_index": np.arange(len(rho_desc)),
                "rho": rho_desc,
                "rho shift": rho_surrogate - rho_desc,
                "surrogate error [%]": 100.0 * np.abs(shifted - exact) / np.abs(exact),
            }
        )
        for case in CASES:
            local = thomson_points[
                (thomson_points["profile_id"] == run["profile_id"])
                & (thomson_points["inference_case"] == case)
            ].set_index("volume_index")
            relative_sigma = 100.0 * local[f"{symbol}_sigma"] / local[symbol]
            volume_table[f"{LABELS[case]} sigma [%]"] = relative_sigma.reindex(
                volume_table["volume_index"]
            ).to_numpy()
        volume_tables.append(volume_table)
surrogate_error = pd.concat(volume_tables, ignore_index=True)
surrogate_error["error / Poisson sigma"] = (
    surrogate_error["surrogate error [%]"] / surrogate_error["Poisson sigma [%]"]
)
surrogate_error["above 3 Poisson sigma"] = (
    surrogate_error["error / Poisson sigma"] > 3.0
)
surrogate_error.to_csv(
    TABLES_DIRECTORY / "prior_likelihood_posterior_surrogate_error.csv", index=False
)

aggregations = {
    "median error [%]": ("surrogate error [%]", "median"),
    "max error [%]": ("surrogate error [%]", "max"),
    "median Poisson sigma [%]": ("Poisson sigma [%]", "median"),
    "median electronic sigma [%]": ("electronic sigma [%]", "median"),
    "max error / Poisson sigma": ("error / Poisson sigma", "max"),
    "volumes above 3 Poisson sigma": ("above 3 Poisson sigma", "sum"),
}
per_truth = surrogate_error.groupby(["profile", "truth"]).agg(**aggregations)
over_truths = surrogate_error.groupby("profile").agg(**aggregations)
over_truths.index = pd.MultiIndex.from_product([over_truths.index, ["all volumes"]])
display(pd.concat([per_truth, over_truths]).round(3))

# The largest errors, and the uncertainty each noise model gives that volume.
density_error = surrogate_error[surrogate_error["profile"] == "ne"]
worst_volumes = density_error.loc[
    density_error.groupby("truth")["surrogate error [%]"].idxmax(),
    [
        "truth",
        "volume_index",
        "rho",
        "rho shift",
        "surrogate error [%]",
        "Poisson sigma [%]",
        "electronic sigma [%]",
    ],
]
display(worst_volumes.round(4))